# 🎬 Autonomous Trailer Director
### (Swachalit Trailer Nirmaata Agent)

**Uddeshya:** Kisi movie ya web series ke liye bina kisi human editor ke ek compelling aur cinematic **Trailer Strategy & Storyboard** tayaar karna.

## Pipeline (4 Steps)

| Step | Naam | Kya hota hai |
|---|---|---|
| 1 | **Content Analysis** | Script + dialogues + character details parse karke har scene ko score karna |
| 2 | **Scene Selection** | Suspense / emotion / action ke hisaab se best scenes chunna (spoilers avoid karke) |
| 3 | **Pacing & Sequencing** | 4-act trailer structure, tempo curve, cut-lengths |
| 4 | **Output Generation** | Timeline, timestamps, music/SFX cues, visual descriptions → `.json` + `.md` |

```
Script/SRT ─► [1 Analyze] ─► [2 Select] ─► [3 Sequence] ─► [4 Storyboard .json/.md]
                 │               │              │
            scene scores    spoiler filter   tempo curve + music cues
```

> **Note:** Notebook default mein **offline (rule-based)** chalta hai, koi API key nahi chahiye. Step 9 mein optional LLM polish (Claude) bhi hai.

---
## Step 0 — Setup
### 0.1 Libraries

In [ ]:
# !pip install numpy pandas matplotlib   # (zaroorat ho to uncomment karein)
import re, json, math, os
from dataclasses import dataclass, field, asdict
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 160)
print("Setup OK")

### 0.2 Configuration (yahin se saari tuning hoti hai)
- `target_trailer_sec` → trailer ki total length
- `n_scenes_to_pick` → kitne scenes trailer mein jayenge
- `weights` → scene score mein suspense/action/emotion ka weightage
- `spoiler_*` → climax / twist wale scenes ko block karne ke rules

In [ ]:
CONFIG = {
    "target_trailer_sec": 60,        # total trailer length (seconds)
    "title_card_sec": 5,             # end title card
    "n_scenes_to_pick": 7,           # scenes in trailer

    # scoring weights (sum ~ 1.0)
    "weights": {"suspense": 0.30, "action": 0.25, "emotion": 0.25,
                "character": 0.10, "quotability": 0.10},
    "peak_bonus": 0.15,              # reward scenes that are strongly one type

    # spoiler control
    "spoiler_tail_fraction": 0.85,   # last 15% of runtime = climax zone
    "spoiler_penalty": 0.60,
    "spoiler_block_threshold": 0.70, # >= this => scene excluded

    # diversity
    "diversity_penalty_type": 0.12,
    "diversity_penalty_location": 0.08,

    # output
    "output_dir": "outputs",
    "use_llm": False,                # Step 9 (optional)
    "llm_model": "claude-sonnet-5-5",
}
OUT = Path(CONFIG["output_dir"]); OUT.mkdir(exist_ok=True)
print(json.dumps({k: v for k, v in CONFIG.items() if k != "weights"}, indent=2))

### 0.3 Helper functions (timecode conversion)

In [ ]:
def to_sec(tc: str) -> int:
    """'00:14:10' -> 850"""
    h, m, s = [int(x) for x in tc.split(":")]
    return h * 3600 + m * 60 + s

def to_tc(sec: float, ms: bool = True) -> str:
    """850.5 -> '00:14:10.5'"""
    sec = max(0.0, float(sec))
    h, m, s = int(sec // 3600), int((sec % 3600) // 60), sec % 60
    return f"{h:02d}:{m:02d}:{s:04.1f}" if ms else f"{h:02d}:{m:02d}:{int(s):02d}"

print(to_sec("00:14:10"), to_tc(850.5), to_tc(7.25))

---
# STEP 1 — Content Analysis
Goal: poora script, dialogue transcript aur character details ko structured data mein badalna, phir har scene ka emotional "fingerprint" nikalna.

## 1.1 Input data
Hum ek **sample web-series episode** ("Raat Ka Saaya", 46 min thriller) use kar rahe hain.
Format:
```
SCENE <no> | <start>-<end> | <HEADING>
ACTION: ...
CHARACTER: dialogue
```
Apna script use karna ho to **1.6** dekhein.

In [ ]:
SAMPLE_SCRIPT = """
TITLE: Raat Ka Saaya
GENRE: Thriller
LOGLINE: Ek bhai apni gayab behen ko dhoondhne ek shraapit haveli mein aata hai, jahan har kamre ka apna ek raaz hai.

SCENE 1 | 00:01:30-00:03:10 | EXT. HAVELI GATE - DUSK
ACTION: Ravi ek purane lohe ke gate ke saamne rukta hai. Hawa mein sannata hai.
ACTION: Gate khud-ba-khud chargraata hua khulta hai.
RAVI: Meera yahin aayi thi. Teen din se uski koi khabar nahi.
THAKUR: Is haveli mein jo aata hai, wapas nahi jaata.

SCENE 2 | 00:04:20-00:06:00 | INT. HAVELI HALL - NIGHT
ACTION: Lambi dining table par sab chup hain. Thakur dheere se muskurata hai.
ACTION: Kaali ke haath se plate girti hai. Poora hall sannate mein doob jaata hai.
KAALI: Raat ko neeche mat jaana, sahab.
THAKUR: Kaali! Chup raho!

SCENE 3 | 00:09:00-00:10:30 | INT. KITCHEN - NIGHT
ACTION: Ravi akela baitha hai, Meera ki purani photo haath mein hai. Uski aankhon mein aansu hain.
RAVI: Maine usse promise kiya tha, hamesha uska saath dunga.
RAVI: Maaf kar de Meera, mujhe bahut der ho gayi.

SCENE 4 | 00:14:10-00:16:00 | INT. BASEMENT - NIGHT
ACTION: Ravi ko deewar ke peeche ek hidden darwaza milta hai. Andhera, footsteps aur ek whisper sunai deta hai.
ACTION: Torch bujh jaati hai. Peeche se ek cheekh goonjti hai.
RAVI: Koi hai? Kaun hai wahan?
VOICE: Wapas ja... abhi bhi waqt hai.

SCENE 5 | 00:19:40-00:21:00 | INT. POLICE STATION - DAY
ACTION: Inspector file band karke meez par patakta hai. Ravi gusse mein hai.
INSPECTOR: Case band ho chuka hai, Ravi. Ghar wapas jao.
RAVI: Meri behen ka case band nahi hua hai!

SCENE 6 | 00:24:00-00:26:30 | EXT. FOREST - NIGHT
ACTION: Naqaabposh log Ravi ke peeche bhaag rahe hain. Gunshot goonjta hai. Ravi khai ke upar se jump karta hai.
ACTION: Chase ke beech ek gaadi ka crash hota hai, aag ki lapten uthti hain.
RAVI: Bhaag! Rukna mat!

SCENE 7 | 00:29:30-00:31:00 | INT. TEMPLE - DAY
ACTION: Kaali mandir mein ghutno ke bal baithi hai, dheemi awaaz mein ek raaz batati hai.
KAALI: Meera ke baare mein sach is haveli ki neev mein dafan hai.
RAVI: Tum yeh sab pehle kyun nahi bataya?

SCENE 8 | 00:33:00-00:34:30 | EXT. RAILWAY STATION - DAY
ACTION: Ravi ghar wapas jaane ka tikat haath mein leke khada hai. Bheed mein se koi usse ghoor raha hai.
ACTION: Ek parchhai uski taraf badhti hai, train ki siti goonjti hai.
RAVI: Yeh meri behen ki handwriting hai... usne mujhe bulaya tha.

SCENE 9 | 00:36:00-00:38:00 | INT. LIBRARY - NIGHT
ACTION: Library mein aag lag jaati hai. Ravi aur Thakur ke guards ke beech zabardast fight hoti hai.
ACTION: Ek dhamaka hota hai aur kitabon ki almari Ravi par gir padti hai.
RAVI: Mujhe diary chahiye, chahe jaan chali jaaye!
GUARD: Maar do isse!

SCENE 10 | 00:39:30-00:41:30 | INT. ROOFTOP - NIGHT
ACTION: Thakur aur Ravi chhat ke kinare par aamne-saamne hain. Tez hawa chal rahi hai.
THAKUR: Asli qaatil main nahi, Inspector hai. Tum shuru se galat the.
RAVI: Tum jhooth bol rahe ho!

SCENE 11 | 00:42:00-00:43:30 | INT. SECRET CHAMBER - NIGHT
ACTION: Ravi ko Meera mil jaati hai. Aansuon ke saath ek emotional reunion hota hai, dono gale lagte hain.
MEERA: Bhaiya... tum aa gaye.

SCENE 12 | 00:44:00-00:45:30 | EXT. HAVELI - DAWN
ACTION: Haveli burns down. Thakur aag mein dies karta hai jabki Ravi aur Meera bahar nikalte hain.
RAVI: Ab sab khatam ho gaya.
"""

CHARACTERS = {
    "Ravi":      {"role": "Protagonist - bhai jo behen ko dhoondh raha hai", "weight": 1.0},
    "Meera":     {"role": "Gayab behen, kahani ka emotional centre",         "weight": 0.8},
    "Thakur":    {"role": "Antagonist - haveli ka maalik",                   "weight": 0.9},
    "Kaali":     {"role": "Rahasyamayi naukrani, raaz jaanti hai",           "weight": 0.6},
    "Inspector": {"role": "Police officer - supporting",                     "weight": 0.5},
}
print("Script chars:", len(SAMPLE_SCRIPT), "| Characters:", list(CHARACTERS))

## 1.2 Parser — script ko `Scene` objects mein todna
Har scene ke liye: number, start/end timecode, location, time-of-day, action lines, dialogue lines.

In [ ]:
SCENE_RE = re.compile(
    r"^SCENE\s+(\d+)\s*\|\s*(\d{2}:\d{2}:\d{2})\s*-\s*(\d{2}:\d{2}:\d{2})\s*\|\s*(.+)$")
META_KEYS = {"TITLE", "GENRE", "LOGLINE", "SETTING"}

@dataclass
class Scene:
    idx: int
    start: int
    end: int
    heading: str
    location: str
    time_of_day: str
    actions: list = field(default_factory=list)
    dialogues: list = field(default_factory=list)   # [(speaker, line), ...]

    @property
    def duration(self): return self.end - self.start
    @property
    def text(self): return " ".join(self.actions + [l for _, l in self.dialogues])

def parse_script(raw: str):
    meta, scenes, cur = {}, [], None
    for line in raw.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        m = SCENE_RE.match(line)
        if m:
            idx, s, e, head = m.groups()
            parts = [p.strip() for p in head.split(" - ")]
            loc = re.sub(r"^(INT|EXT)\.\s*", "", parts[0])
            tod = parts[-1] if len(parts) > 1 else ""
            cur = Scene(int(idx), to_sec(s), to_sec(e), head, loc, tod)
            scenes.append(cur)
            continue
        key, _, val = line.partition(":")
        key, val = key.strip(), val.strip()
        if cur is None:
            if key.upper() in META_KEYS:
                meta[key.lower()] = val
            continue
        if key.upper() == "ACTION":
            cur.actions.append(val)
        elif re.fullmatch(r"[A-Z][A-Z .]{1,20}", key):
            cur.dialogues.append((key.title(), val))
    return meta, scenes

meta, scenes = parse_script(SAMPLE_SCRIPT)
print(meta["title"], "|", meta["genre"], "|", len(scenes), "scenes")
print(scenes[3])

## 1.3 Parsed scenes ka overview table

In [ ]:
EP_END = max(s.end for s in scenes)

overview = pd.DataFrame([{
    "scene": s.idx, "heading": s.heading, "start": to_tc(s.start, False),
    "end": to_tc(s.end, False), "dur_s": s.duration,
    "n_actions": len(s.actions), "n_dialogues": len(s.dialogues),
    "words": len(s.text.split()),
} for s in scenes]).set_index("scene")
print(f"Episode length: {to_tc(EP_END, False)}")
overview

## 1.4 Emotion Lexicons — suspense / emotion / action / spoiler
Hinglish + English dono ke keywords. Yeh "lightweight NLP" hai; Step 9 mein isko LLM se replace/enrich kar sakte hain.

In [ ]:
LEX = {
    "suspense": ["chup", "andhera", "dark", "whisper", "footstep", "parchhai", "shadow", "raaz",
                 "secret", "gayab", "missing", "khabar", "darr", "fear", "scream", "cheekh",
                 "hidden", "khauf", "wapas nahi", "mat jaana", "koi hai", "sannata", "darwaza",
                 "dafan", "ghoor", "neev", "chargraata", "bujh"],
    "emotion":  ["promise", "aansu", "aansuon", "tears", "cry", "gale", "hug", "maaf", "sorry",
                 "pyaar", "love", "yaad", "behen", "bhaiya", "sister", "dil", "saath",
                 "photo", "handwriting", "bulaya", "emotional"],
    "action":   ["bhaag", "bhaagna", "chase", "gunshot", "fight", "dhamaka", "explosion", "aag",
                 "fire", "jump", "crash", "naqaabposh", "maar", "guards", "gir", "zabardast",
                 "gaadi", "khai", "attack", "tez"],
    # Spoiler: story ka twist / resolution reveal karne wale words
    "spoiler":  ["asli qaatil", "qaatil", "dies", "mar gaya", "reunion", "burns down",
                 "khatam ho gaya", "raakh"],
}

def count_hits(text: str, terms) -> int:
    t = text.lower()
    return sum(len(re.findall(r"\b" + re.escape(w), t)) for w in terms)

print(count_hits("Andhera hai, koi hai? Cheekh sunai di!", LEX["suspense"]))

## 1.5 Per-scene feature extraction
Har scene ke liye:
- **raw scores** (keyword density per 100 words)
- **intensity** (`!` aur `?` ka density)
- **character presence** (kaun-kaun se characters scene mein hain, weighted)
- **quotability** (kya koi short, punchy dialogue hai jo trailer VO ban sake?)
- **spoiler risk** (twist words + climax zone)

In [ ]:
def best_quote(sc: Scene):
    """Trailer-worthy dialogue: 4-14 words, spoiler-free, lexicon hits = bonus."""
    best = None
    for spk, line in sc.dialogues:
        n = len(line.split())
        if not (4 <= n <= 14) or count_hits(line, LEX["spoiler"]) > 0:
            continue
        hits = sum(count_hits(line, LEX[k]) for k in ("suspense", "emotion", "action"))
        punct = 0.1 if line.rstrip().endswith(("!", "?", "...")) else 0.0
        score = min(1.0, 0.3 + 0.2 * hits + punct)
        if best is None or score > best[2]:
            best = (spk, line, score)
    return best   # (speaker, line, score) or None

def present_characters(sc: Scene):
    speakers = {s for s, _ in sc.dialogues}
    mentioned = {c for c in CHARACTERS if re.search(r"\b" + c + r"\b", sc.text)}
    return sorted((speakers | mentioned) & set(CHARACTERS))

def spoiler_risk(sc: Scene, cfg=CONFIG):
    hits = count_hits(sc.text, LEX["spoiler"])
    risk = 0.0
    if hits:
        risk += 0.5 + 0.2 * min(hits, 2)
    if sc.start / EP_END >= cfg["spoiler_tail_fraction"]:
        risk += 0.4
    return min(1.0, risk)

def extract_features(scenes):
    rows = []
    for s in scenes:
        words = max(len(s.text.split()), 1)
        q = best_quote(s)
        chars = present_characters(s)
        rows.append({
            "scene": s.idx, "heading": s.heading, "location": s.location,
            "start": s.start, "end": s.end, "dur": s.duration,
            "raw_suspense": count_hits(s.text, LEX["suspense"]) / words * 100,
            "raw_emotion":  count_hits(s.text, LEX["emotion"])  / words * 100,
            "raw_action":   count_hits(s.text, LEX["action"])   / words * 100,
            "raw_intensity": (s.text.count("!") + s.text.count("?")) / words * 100,
            "char_weight": sum(CHARACTERS[c]["weight"] for c in chars),
            "characters": ", ".join(chars),
            "quotability": q[2] if q else 0.0,
            "spoiler_risk": spoiler_risk(s),
        })
    return pd.DataFrame(rows).set_index("scene")

feat = extract_features(scenes)
feat[["heading", "raw_suspense", "raw_emotion", "raw_action", "raw_intensity",
      "quotability", "spoiler_risk"]].round(2)

## 1.6 Normalization — sab scores ko 0–1 range mein laana
Taaki suspense, emotion, action ko aapas mein compare kiya ja sake. Saath mein har scene ka **dominant type** bhi nikalte hain.

In [ ]:
def minmax(s: pd.Series) -> pd.Series:
    rng = s.max() - s.min()
    return (s - s.min()) / rng if rng > 0 else s * 0.0

def normalize(feat: pd.DataFrame) -> pd.DataFrame:
    df = feat.copy()
    for k in ("suspense", "emotion", "action", "intensity"):
        df[k] = minmax(df[f"raw_{k}"])
    df["character"] = minmax(df["char_weight"])
    df["dominant"] = df[["suspense", "emotion", "action"]].idxmax(axis=1)
    return df

df = normalize(feat)
df[["heading", "suspense", "emotion", "action", "intensity", "character",
    "dominant", "spoiler_risk"]].round(2)

## 1.7 Character analysis — kaun kahani pe haavi hai?
Dialogue lines aur scene appearances ginte hain.

In [ ]:
line_ct = Counter(spk for s in scenes for spk, _ in s.dialogues)
scene_ct = Counter(c for s in scenes for c in present_characters(s))
char_df = pd.DataFrame({
    "role": {c: v["role"] for c, v in CHARACTERS.items()},
    "weight": {c: v["weight"] for c, v in CHARACTERS.items()},
    "dialogue_lines": {c: line_ct.get(c, 0) for c in CHARACTERS},
    "scenes_present": {c: scene_ct.get(c, 0) for c in CHARACTERS},
}).sort_values("scenes_present", ascending=False)
char_df

## 1.8 (Optional) Apna script / SRT use karna
- **Same format** wali `.txt` file → `parse_script(open(path).read())`
- **Sirf subtitle (.srt)** ho to niche wala function use karein → yeh transcript ko fixed time-windows mein "pseudo-scenes" bana deta hai.

In [ ]:
def scenes_from_srt(path: str, window: int = 90):
    """.srt -> list[Scene] (har `window` seconds ka ek pseudo-scene)."""
    raw = Path(path).read_text(encoding="utf-8", errors="ignore")
    pat = re.compile(r"(\d+):(\d+):(\d+)[,.]\d+\s*-->\s*(\d+):(\d+):(\d+)[,.]\d+\s*\n(.*?)(?=\n\s*\n|\Z)",
                     re.S)
    buckets = {}
    for m in pat.finditer(raw):
        h, mi, se, h2, m2, s2 = map(int, m.groups()[:6])
        start = h * 3600 + mi * 60 + se
        text = " ".join(m.group(7).split())
        buckets.setdefault(start // window, []).append((start, text))
    out = []
    for i, (b, items) in enumerate(sorted(buckets.items()), 1):
        sc = Scene(i, b * window, (b + 1) * window, f"SEGMENT {i}", f"SEGMENT {i}", "")
        sc.dialogues = [("Speaker", t) for _, t in items]
        out.append(sc)
    return out

# Example (uncomment):
# scenes = scenes_from_srt("my_episode.srt"); EP_END = max(s.end for s in scenes)
# feat = extract_features(scenes); df = normalize(feat)
print("SRT loader ready")

---
# STEP 2 — Scene Selection
Goal: trailer ke liye sabse **suspenseful / emotional / action-packed** scenes chunna, lekin **spoilers ke bina** aur **variety** ke saath.

## 2.1 Trailer-worthiness score
$$score = \sum_k w_k \cdot f_k \; + \; peak\_bonus \cdot \max(susp, emo, act) \; - \; spoiler\_penalty \cdot risk$$

In [ ]:
def trailer_score(df: pd.DataFrame, cfg=CONFIG) -> pd.DataFrame:
    w = cfg["weights"]
    d = df.copy()
    base = (w["suspense"] * d["suspense"] + w["action"] * d["action"] +
            w["emotion"] * d["emotion"] + w["character"] * d["character"] +
            w["quotability"] * d["quotability"])
    peak = cfg["peak_bonus"] * d[["suspense", "emotion", "action"]].max(axis=1)
    d["trailer_score"] = (base + peak - cfg["spoiler_penalty"] * d["spoiler_risk"]).clip(lower=0)
    return d

scored = trailer_score(df)
scored[["heading", "dominant", "spoiler_risk", "trailer_score"]] \
    .sort_values("trailer_score", ascending=False).round(3)

## 2.2 Spoiler filter
`spoiler_risk >= threshold` wale scenes (twist/climax/resolution) trailer se **hard-block** ho jaate hain.

In [ ]:
blocked = scored[scored["spoiler_risk"] >= CONFIG["spoiler_block_threshold"]]
eligible = scored.drop(blocked.index)
print(f"Blocked (spoiler): {list(blocked.index)}  |  Eligible: {list(eligible.index)}")
blocked[["heading", "spoiler_risk"]]

## 2.3 Diversity-aware greedy selection
Sirf top-score lene se saare scenes ek hi type ke ho sakte hain. Isliye greedy loop mein:
- ek hi **dominant type** ke 2 se zyada scenes pe penalty
- ek hi **location** repeat hone pe penalty

In [ ]:
def select_scenes(scored: pd.DataFrame, cfg=CONFIG):
    pool = scored[scored["spoiler_risk"] < cfg["spoiler_block_threshold"]].copy()
    picked, type_cnt, loc_seen, log = [], Counter(), set(), []
    while len(picked) < cfg["n_scenes_to_pick"] and len(pool):
        adj = pool["trailer_score"].copy()
        adj -= pool["dominant"].map(lambda t: cfg["diversity_penalty_type"] * max(0, type_cnt[t] - 2))
        adj -= pool["location"].map(lambda l: cfg["diversity_penalty_location"] if l in loc_seen else 0)
        best = adj.idxmax()
        log.append((best, round(float(adj[best]), 3)))
        picked.append(best)
        type_cnt[pool.loc[best, "dominant"]] += 1
        loc_seen.add(pool.loc[best, "location"])
        pool = pool.drop(best)
    return picked, log

picked, sel_log = select_scenes(scored)
selected = scored.loc[picked]
print("Selected scene order of picking:", picked)
selected[["heading", "dominant", "trailer_score"]].round(3)

## 2.4 Selection report — kya chuna, kya chhoda, kyun
Yeh explainability ke liye important hai (judges / client ko dikhane ke kaam aata hai).

In [ ]:
def reason(row, chosen: bool):
    if row["spoiler_risk"] >= CONFIG["spoiler_block_threshold"]:
        return "Blocked: spoiler/twist/climax zone"
    if chosen:
        return f"Picked: strong {row['dominant']} (score {row['trailer_score']:.2f})"
    return f"Skipped: lower score/diversity (score {row['trailer_score']:.2f})"

selection_report = pd.DataFrame({
    "heading": scored["heading"],
    "dominant": scored["dominant"],
    "score": scored["trailer_score"].round(3),
    "decision": [reason(r, i in picked) for i, r in scored.iterrows()],
})
selection_report

---
# STEP 3 — Cinematic Pacing & Sequencing
Goal: chune hue scenes ko aise order mein lagana ki **tension build ho** aur trailer ka tempo dheere-dheere tez hota jaaye.

## 3.1 4-Act Trailer Structure
| Act | Kaam | Tempo | Cuts/scene | Cut length |
|---|---|---|---|---|
| **HOOK** | Mystery se pakadna | Slow | 1 | ~5 s |
| **SETUP** | Duniya + stakes + emotion | Medium | 2 | ~3.5 s |
| **ESCALATION** | Khatra badhna | Building | 2 | ~2.5 s |
| **MONTAGE** | Fast climax-style cuts | Fast | 3 | ~1.5 s |
| **TITLE CARD** | Naam + tagline + stinger | – | – | fixed |

In [ ]:
ACTS = {
    "HOOK": dict(label="Cold Open / Hook", tempo="slow", cuts=1, cut_len=5.0, bpm=60,
                 camera="Slow push-in, wide static frames, shallow depth of field",
                 transition="Slow fade from black",
                 music="Sparse low drone + distant heartbeat, a single piano note every 4 beats"),
    "SETUP": dict(label="World & Stakes", tempo="medium", cuts=2, cut_len=3.5, bpm=80,
                  camera="Handheld close-ups, warm-to-cold colour shift",
                  transition="Cross dissolve",
                  music="Cello motif enters, soft pulse builds under the dialogue"),
    "ESCALATION": dict(label="Escalation", tempo="building", cuts=2, cut_len=2.5, bpm=100,
                       camera="Tracking shots, dutch angles, tightening framing",
                       transition="Hard cut on beat",
                       music="Rising strings (shepard tone), percussive ticks, riser FX"),
    "MONTAGE": dict(label="Climax Montage", tempo="fast", cuts=3, cut_len=1.5, bpm=130,
                    camera="Rapid whip-pans, quick inserts, strobing light",
                    transition="Smash cut",
                    music="Full orchestral hits + heavy percussion, braams on every cut"),
}
SFX = {
    "suspense": "creaking door, whisper layers, distant footsteps, low rumble",
    "emotion":  "soft breath, room tone, faint sob, muffled piano",
    "action":   "gunshot crack, rapid footsteps, impact hits, debris",
}
pd.DataFrame(ACTS).T[["label", "tempo", "cuts", "cut_len", "bpm"]]

## 3.2 Scenes ko acts mein assign karna
Rules:
1. **HOOK** → sabse suspenseful scene
2. **SETUP** → sabse emotional scenes (~25%)
3. **MONTAGE** → highest action + intensity wale scenes (~50% of rest)
4. **ESCALATION** → bache hue scenes

In [ ]:
def assign_acts(selected: pd.DataFrame):
    remaining = selected.copy()
    hook_idx = remaining["suspense"].idxmax()
    remaining = remaining.drop(hook_idx)

    k = max(1, round(0.25 * len(remaining)))
    setup_idx = remaining.sort_values("emotion", ascending=False).head(k).index.tolist()
    remaining = remaining.drop(setup_idx)

    m = max(2, math.ceil(0.5 * len(remaining)))
    remaining["_heat"] = remaining["action"] + remaining["intensity"]
    montage_idx = remaining.sort_values("_heat", ascending=False).head(m).index.tolist()
    esc_idx = remaining.drop(montage_idx).index.tolist()

    return {"HOOK": [hook_idx], "SETUP": setup_idx, "ESCALATION": esc_idx, "MONTAGE": montage_idx}

act_map = assign_acts(selected)
act_map

## 3.3 Act ke andar ordering → tension curve
- **SETUP** → story-order (taaki context bane)
- **ESCALATION / MONTAGE** → **intensity ascending** (har cut pichle se zyada charged)

In [ ]:
def order_within_acts(act_map, scored):
    ordered = []
    for act in ["HOOK", "SETUP", "ESCALATION", "MONTAGE"]:
        ids = act_map[act]
        if act == "SETUP":
            ids = sorted(ids, key=lambda i: scored.loc[i, "start"])
        elif act in ("ESCALATION", "MONTAGE"):
            ids = sorted(ids, key=lambda i: scored.loc[i, "intensity"] + scored.loc[i, "action"])
        ordered += [(act, i) for i in ids]
    return ordered

sequence = order_within_acts(act_map, scored)
pd.DataFrame([(a, i, scored.loc[i, "heading"], round(scored.loc[i, "intensity"], 2))
              for a, i in sequence], columns=["act", "scene", "heading", "intensity"])

## 3.4 Duration allocation (tempo curve)
Har act ka `cuts × cut_len` raw time hota hai. Phir ek **scale factor** se poore trailer ko `target_trailer_sec − title_card_sec` mein fit karte hain. Cut length act ke saath ghatti jaati hai → **tempo tez hota hai**.

In [ ]:
def allocate_durations(sequence, cfg=CONFIG):
    raw = sum(ACTS[a]["cuts"] * ACTS[a]["cut_len"] for a, _ in sequence)
    budget = cfg["target_trailer_sec"] - cfg["title_card_sec"]
    scale = float(np.clip(budget / raw, 0.6, 1.5))
    return scale, raw, budget

scale, raw_total, budget = allocate_durations(sequence)
print(f"raw={raw_total:.1f}s | budget={budget}s | scale={scale:.2f}")
for act in ACTS:
    print(f"{act:<11} cut_len = {ACTS[act]['cut_len'] * scale:.2f}s  x {ACTS[act]['cuts']} cuts/scene")

## 3.5 Source-window picking
Har scene ke andar se `cut_len` seconds ke **evenly-spaced windows** nikalte hain (taaki ek hi scene ke cuts alag-alag moments dikhayein).

In [ ]:
def source_windows(sc_start, sc_end, n_cuts, cut_len):
    D = sc_end - sc_start
    cut_len = min(cut_len, D / n_cuts)
    wins = []
    for k in range(n_cuts):
        center = sc_start + (k + 0.5) * D / n_cuts
        wins.append((center - cut_len / 2, center + cut_len / 2))
    return wins

print([(to_tc(a), to_tc(b)) for a, b in source_windows(850, 960, 3, 2.0)])

---
# STEP 4 — Output Generation (Storyboard)
Goal: final **Trailer Storyboard** — exact timeline, scene timestamps, audio/music cues, visual description.

## 4.1 Visual description + audio per clip

In [ ]:
def first_sentence(text: str) -> str:
    return re.split(r"(?<=[.!?])\s", text.strip())[0]

def build_clip(order_no, act, scene_idx, t0, scale, scored, scenes_by_idx):
    cfg_act, sc, row = ACTS[act], scenes_by_idx[scene_idx], scored.loc[scene_idx]
    cut_len = cfg_act["cut_len"] * scale
    n = cfg_act["cuts"]
    wins = source_windows(sc.start, sc.end, n, cut_len)
    real_len = (wins[0][1] - wins[0][0])

    cuts, t = [], t0
    for k, (a, b) in enumerate(wins, 1):
        cuts.append({"cut": k, "trailer_in": round(t, 2), "trailer_out": round(t + real_len, 2),
                     "trailer_in_tc": to_tc(t), "trailer_out_tc": to_tc(t + real_len),
                     "source_in_tc": to_tc(a), "source_out_tc": to_tc(b),
                     "duration_s": round(real_len, 2)})
        t += real_len

    q = best_quote(sc)
    visual_src = sc.actions[0] if sc.actions else sc.heading
    clip = {
        "order": order_no, "act": act, "act_label": cfg_act["label"], "tempo": cfg_act["tempo"],
        "source_scene": {"scene_no": sc.idx, "heading": sc.heading,
                         "episode_tc": f"{to_tc(sc.start, False)} - {to_tc(sc.end, False)}"},
        "dominant_mood": row["dominant"], "trailer_score": round(float(row["trailer_score"]), 3),
        "characters": row["characters"],
        "visual_description": f"{cfg_act['camera']}. {first_sentence(visual_src)}",
        "audio": {"music_cue": cfg_act["music"], "bpm": cfg_act["bpm"],
                  "sfx": SFX[row["dominant"]]},
        "dialogue_vo": {"speaker": q[0], "line": q[1]} if q else None,
        "transition_out": cfg_act["transition"],
        "cuts": cuts,
        "clip_start": round(t0, 2), "clip_end": round(t, 2),
    }
    return clip, t

scenes_by_idx = {s.idx: s for s in scenes}
c0, t_end0 = build_clip(1, "HOOK", sequence[0][1], 0.0, scale, scored, scenes_by_idx)
print(json.dumps({k: c0[k] for k in ("act", "visual_description", "audio", "dialogue_vo")}, indent=2, ensure_ascii=False))

## 4.2 Poora timeline build karna
Sab clips ko ek ke baad ek jodte hain; end mein **title card** aata hai.

In [ ]:
def build_timeline(sequence, scale, scored, scenes_by_idx):
    clips, t = [], 0.0
    for n, (act, sidx) in enumerate(sequence, 1):
        clip, t = build_clip(n, act, sidx, t, scale, scored, scenes_by_idx)
        clips.append(clip)
    return clips, t

clips, clips_end = build_timeline(sequence, scale, scored, scenes_by_idx)
print(f"Clips end at {to_tc(clips_end)}  | + title card = {to_tc(clips_end + CONFIG['title_card_sec'])}")

## 4.3 Music cue sheet
Har act ke liye ek music segment + do classic trailer tricks:
- 🔇 **Silence hit** — montage se pehle 0.8s ke liye music cut (sirf heartbeat)
- 💥 **Braam stinger** — title card pe

In [ ]:
def build_music_cues(clips, clips_end, cfg=CONFIG):
    cues = []
    for act in ["HOOK", "SETUP", "ESCALATION", "MONTAGE"]:
        cl = [c for c in clips if c["act"] == act]
        if not cl:
            continue
        s, e = cl[0]["clip_start"], cl[-1]["clip_end"]
        cues.append({"type": "music_segment", "act": act, "start_tc": to_tc(s), "end_tc": to_tc(e),
                     "bpm": ACTS[act]["bpm"], "description": ACTS[act]["music"]})
    mont = [c for c in clips if c["act"] == "MONTAGE"]
    if mont:
        ms = mont[0]["clip_start"]
        cues.append({"type": "silence_hit", "start_tc": to_tc(ms - 0.8), "end_tc": to_tc(ms),
                     "description": "Music drops out completely; only a heartbeat + sub-bass swell before the montage"})
    cues.append({"type": "stinger", "start_tc": to_tc(clips_end),
                 "end_tc": to_tc(clips_end + cfg["title_card_sec"]),
                 "description": "Single deep braam hit with long reverb tail over the title card"})
    return cues

music_cues = build_music_cues(clips, clips_end)
pd.DataFrame(music_cues)[["type", "start_tc", "end_tc", "description"]]

## 4.4 Title card + Logline

In [ ]:
def build_title_card(meta, clips_end, cfg=CONFIG):
    return {"start_tc": to_tc(clips_end), "end_tc": to_tc(clips_end + cfg["title_card_sec"]),
            "title": meta.get("title", "Untitled"),
            "tagline": meta.get("logline", ""),
            "end_line": "Jald hi aa raha hai"}

title_card = build_title_card(meta, clips_end)
title_card

## 4.5 Final Storyboard JSON

In [ ]:
def build_storyboard(meta, clips, clips_end, music_cues, title_card, selection_report, cfg=CONFIG):
    total = clips_end + cfg["title_card_sec"]
    return {
        "meta": {"title": meta.get("title"), "genre": meta.get("genre"),
                 "logline": meta.get("logline"),
                 "target_duration_s": cfg["target_trailer_sec"],
                 "actual_duration_s": round(total, 2),
                 "actual_duration_tc": to_tc(total),
                 "n_clips": len(clips), "n_cuts": sum(len(c["cuts"]) for c in clips)},
        "strategy": [{"act": a, "label": ACTS[a]["label"], "tempo": ACTS[a]["tempo"],
                      "bpm": ACTS[a]["bpm"], "camera_language": ACTS[a]["camera"],
                      "scenes": [c["source_scene"]["scene_no"] for c in clips if c["act"] == a]}
                     for a in ACTS],
        "storyboard": clips,
        "music_cues": music_cues,
        "title_card": title_card,
        "selection_report": selection_report.reset_index().to_dict(orient="records"),
    }

storyboard = build_storyboard(meta, clips, clips_end, music_cues, title_card, selection_report)
print(json.dumps(storyboard["meta"], indent=2, ensure_ascii=False))

## 4.6 Markdown storyboard (human-readable)

In [ ]:
def render_markdown(sb) -> str:
    m = sb["meta"]
    L = [f"# 🎬 Trailer Storyboard — {m['title']}",
         f"**Genre:** {m['genre']}  |  **Duration:** {m['actual_duration_tc']}  |  "
         f"**Clips:** {m['n_clips']}  |  **Cuts:** {m['n_cuts']}", "",
         f"> {m['logline']}", "", "## Trailer Strategy", "",
         "| Act | Tempo | BPM | Camera language | Source scenes |", "|---|---|---|---|---|"]
    for a in sb["strategy"]:
        L.append(f"| {a['label']} | {a['tempo']} | {a['bpm']} | {a['camera_language']} | "
                 f"{', '.join(map(str, a['scenes']))} |")
    L += ["", "## Timeline", ""]
    for c in sb["storyboard"]:
        s = c["source_scene"]
        L += [f"### {c['order']}. {c['act_label']} — Scene {s['scene_no']} ({s['heading']})",
              f"- **Trailer time:** {c['cuts'][0]['trailer_in_tc']} → {c['cuts'][-1]['trailer_out_tc']}",
              f"- **Episode timestamps:** " + "; ".join(f"{x['source_in_tc']}–{x['source_out_tc']}" for x in c["cuts"]),
              f"- **Mood:** {c['dominant_mood']}  |  **Characters:** {c['characters']}",
              f"- **Visual:** {c['visual_description']}",
              f"- **Music:** {c['audio']['music_cue']} ({c['audio']['bpm']} BPM)",
              f"- **SFX:** {c['audio']['sfx']}"]
        if c["dialogue_vo"]:
            L.append(f"- **VO / Dialogue:** {c['dialogue_vo']['speaker']}: \"{c['dialogue_vo']['line']}\"")
        L += [f"- **Transition out:** {c['transition_out']}", ""]
    L += ["## Music Cue Sheet", "", "| Type | Start | End | Description |", "|---|---|---|---|"]
    for q in sb["music_cues"]:
        L.append(f"| {q['type']} | {q['start_tc']} | {q['end_tc']} | {q['description']} |")
    t = sb["title_card"]
    L += ["", "## Title Card", f"- **{t['start_tc']} → {t['end_tc']}**: *{t['title']}*",
          f"- Tagline: {t['tagline']}", f"- End line: {t['end_line']}", "",
          "## Selection Report", "", "| Scene | Heading | Decision |", "|---|---|---|"]
    for r in sb["selection_report"]:
        L.append(f"| {r['scene']} | {r['heading']} | {r['decision']} |")
    return "\n".join(L)

md_text = render_markdown(storyboard)
print(md_text[:1800])

---
# STEP 5 — Visualization
Dekhte hain scene scores, selection aur final tempo curve.

## 5.1 Scene scores + selection

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
x = np.arange(len(scored)); w = 0.26
ax.bar(x - w, scored["suspense"], w, label="Suspense", color="#6c5ce7")
ax.bar(x,     scored["emotion"],  w, label="Emotion",  color="#e17055")
ax.bar(x + w, scored["action"],   w, label="Action",   color="#00b894")
ax.plot(x, scored["trailer_score"], "k.-", label="Trailer score")
for i, idx in enumerate(scored.index):
    if idx in picked:
        ax.axvspan(i - 0.45, i + 0.45, color="gold", alpha=0.25)
    if idx in blocked.index:
        ax.text(i, 1.02, "spoiler", ha="center", fontsize=7, color="crimson", rotation=90)
ax.set_xticks(x); ax.set_xticklabels([f"S{i}" for i in scored.index])
ax.set_ylim(0, 1.25); ax.set_title("Scene scores (gold = selected, red = spoiler-blocked)")
ax.legend(ncol=4, fontsize=8, loc="upper left"); plt.tight_layout(); plt.show()

## 5.2 Trailer tempo curve (cut length vs time)
Cut length ghatna = tempo badhna. Background colours acts dikhate hain.

In [ ]:
colors = {"HOOK": "#dfe6e9", "SETUP": "#ffeaa7", "ESCALATION": "#fab1a0", "MONTAGE": "#ff7675"}
fig, ax = plt.subplots(figsize=(11, 3.6))
for c in clips:
    ax.axvspan(c["clip_start"], c["clip_end"], color=colors[c["act"]], alpha=0.6)
xs, ys = [], []
for c in clips:
    for k in c["cuts"]:
        xs += [k["trailer_in"], k["trailer_out"]]; ys += [k["duration_s"]] * 2
ax.step(xs, ys, where="post", color="black")
ax.set_xlabel("Trailer time (s)"); ax.set_ylabel("Cut length (s)")
ax.set_title("Tempo curve - cuts get shorter as the trailer builds")
handles = [plt.Rectangle((0, 0), 1, 1, color=v) for v in colors.values()]
ax.legend(handles, colors.keys(), ncol=4, fontsize=8); plt.tight_layout(); plt.show()

## 5.3 Storyboard cuts table

In [ ]:
cut_rows = []
for c in clips:
    for k in c["cuts"]:
        cut_rows.append({"trailer_in": k["trailer_in_tc"], "trailer_out": k["trailer_out_tc"],
                         "act": c["act"], "scene": c["source_scene"]["scene_no"],
                         "source_in": k["source_in_tc"], "source_out": k["source_out_tc"],
                         "dur_s": k["duration_s"]})
cuts_df = pd.DataFrame(cut_rows)
cuts_df

---
# STEP 6 — Validation (Quality Checks)
Automatic checks jo confirm karte hain ki storyboard "editor-ready" hai.

In [ ]:
def validate(sb, cfg=CONFIG):
    checks = []
    total = sb["meta"]["actual_duration_s"]
    checks.append(("Duration within +/-8% of target",
                   abs(total - cfg["target_trailer_sec"]) / cfg["target_trailer_sec"] <= 0.08,
                   f"{total:.1f}s vs {cfg['target_trailer_sec']}s"))

    risky = [c["source_scene"]["scene_no"] for c in sb["storyboard"]
             if scored.loc[c["source_scene"]["scene_no"], "spoiler_risk"] >= cfg["spoiler_block_threshold"]]
    checks.append(("No spoiler scenes used", not risky, f"risky={risky}"))

    flat = [k for c in sb["storyboard"] for k in c["cuts"]]
    contiguous = all(abs(flat[i]["trailer_out"] - flat[i + 1]["trailer_in"]) < 0.02 for i in range(len(flat) - 1))
    checks.append(("Timeline contiguous (no gaps/overlaps)", contiguous, f"{len(flat)} cuts"))

    in_range = all(s["source_in_tc"] <= s["source_out_tc"] for s in flat)
    checks.append(("Source in < out for every cut", in_range, ""))

    acts_present = {c["act"] for c in sb["storyboard"]}
    checks.append(("All 4 acts present", acts_present == set(ACTS), str(sorted(acts_present))))

    avg = {a: np.mean([k["duration_s"] for c in sb["storyboard"] if c["act"] == a for k in c["cuts"]])
           for a in ACTS if a in acts_present}
    vals = list(avg.values())
    checks.append(("Tempo accelerates (avg cut length decreases)",
                   all(vals[i] >= vals[i + 1] for i in range(len(vals) - 1)),
                   ", ".join(f"{a}={v:.1f}s" for a, v in avg.items())))

    n_vo = sum(1 for c in sb["storyboard"] if c["dialogue_vo"])
    checks.append(("At least 3 dialogue/VO hooks", n_vo >= 3, f"{n_vo} clips"))

    return pd.DataFrame(checks, columns=["check", "passed", "detail"])

report = validate(storyboard)
report["passed"] = report["passed"].map({True: "PASS", False: "FAIL"})
report

---
# STEP 7 — Save Outputs (`.json` + `.md`)

In [ ]:
json_path = OUT / "trailer_storyboard.json"
md_path = OUT / "trailer_storyboard.md"
json_path.write_text(json.dumps(storyboard, indent=2, ensure_ascii=False), encoding="utf-8")
md_path.write_text(md_text, encoding="utf-8")
cuts_df.to_csv(OUT / "edit_decision_list.csv", index=False)
print("Saved:", json_path, "|", md_path, "|", OUT / "edit_decision_list.csv")

---
# STEP 8 — One-call Pipeline (kisi bhi naye script ke liye)
Upar ke saare substeps ek function mein. Naya script do, poora storyboard le lo.

In [ ]:
def run_pipeline(raw_script: str, characters: dict = None, cfg: dict = None, save=True):
    """raw_script (SCENE format) -> storyboard dict + markdown"""
    global CHARACTERS, EP_END, CONFIG, scored, blocked, picked
    if characters: CHARACTERS = characters
    if cfg: CONFIG = {**CONFIG, **cfg}

    meta, scn = parse_script(raw_script)                                   # 1.2
    EP_END = max(s.end for s in scn)
    d = normalize(extract_features(scn))                                    # 1.5-1.6
    scored = trailer_score(d, CONFIG)                                       # 2.1
    blocked = scored[scored["spoiler_risk"] >= CONFIG["spoiler_block_threshold"]]  # 2.2
    picked, _ = select_scenes(scored, CONFIG)                               # 2.3
    sel = scored.loc[picked]
    seq = order_within_acts(assign_acts(sel), scored)                       # 3.2-3.3
    scl, _, _ = allocate_durations(seq, CONFIG)                             # 3.4
    by_idx = {s.idx: s for s in scn}
    cl, end = build_timeline(seq, scl, scored, by_idx)                      # 4.2
    rep = pd.DataFrame({"heading": scored["heading"], "dominant": scored["dominant"],
                        "score": scored["trailer_score"].round(3),
                        "decision": [reason(r, i in picked) for i, r in scored.iterrows()]})
    sb = build_storyboard(meta, cl, end, build_music_cues(cl, end, CONFIG),
                          build_title_card(meta, end, CONFIG), rep, CONFIG)   # 4.3-4.5
    text = render_markdown(sb)                                              # 4.6
    if save:
        (OUT / "trailer_storyboard.json").write_text(json.dumps(sb, indent=2, ensure_ascii=False), encoding="utf-8")
        (OUT / "trailer_storyboard.md").write_text(text, encoding="utf-8")
    return sb, text

sb2, md2 = run_pipeline(SAMPLE_SCRIPT, save=False)
print("Pipeline OK ->", sb2["meta"]["actual_duration_tc"], "|", sb2["meta"]["n_clips"], "clips")

---
# STEP 9 — (Optional) LLM Polish with Claude
Rule-based pipeline offline chalta hai. Agar aap chahte hain ki **visual descriptions aur logline** aur zyada cinematic ho jayein, to Claude se rewrite karwa sakte hain.

Setup:
```bash
pip install anthropic
export ANTHROPIC_API_KEY="sk-ant-..."
```
`CONFIG["use_llm"] = True` karke niche wala cell chalayein. Fail hone par original storyboard waisa hi rehta hai.

In [ ]:
def polish_with_llm(sb: dict, model: str = CONFIG["llm_model"]) -> dict:
    """Visual descriptions ko cinematic Hinglish mein rewrite karta hai (structure/timestamps untouched)."""
    try:
        import anthropic
        client = anthropic.Anthropic()          # ANTHROPIC_API_KEY env se
    except Exception as e:
        print("LLM skipped:", e); return sb

    payload = [{"order": c["order"], "act": c["act"], "scene": c["source_scene"]["heading"],
                "draft": c["visual_description"], "mood": c["dominant_mood"]} for c in sb["storyboard"]]
    prompt = ("Tum ek trailer editor ho. Neeche har clip ka draft visual description hai. "
              "Har ek ko 1-2 vivid, cinematic Hinglish sentences mein rewrite karo (camera + lighting + emotion). "
              "Sirf valid JSON list return karo: [{\"order\": int, \"visual\": str}].\n\n"
              + json.dumps(payload, ensure_ascii=False))
    try:
        resp = client.messages.create(model=model, max_tokens=1500,
                                      messages=[{"role": "user", "content": prompt}])
        txt = "".join(b.text for b in resp.content if b.type == "text")
        data = json.loads(re.sub(r"```json|```", "", txt).strip())
        by_order = {d["order"]: d["visual"] for d in data}
        for c in sb["storyboard"]:
            if c["order"] in by_order:
                c["visual_description_polished"] = by_order[c["order"]]
        print("LLM polish applied to", len(by_order), "clips")
    except Exception as e:
        print("LLM polish failed (original kept):", e)
    return sb

if CONFIG["use_llm"]:
    storyboard = polish_with_llm(storyboard)
else:
    print("use_llm=False -> offline output hi final hai.")

---
# ✅ Final Output Preview

In [ ]:
from IPython.display import Markdown, display
display(Markdown(md_text))

---
## Next Steps / Improvements
1. **Embeddings** se scene similarity (diversity ke liye) — `sentence-transformers`
2. **Audio-visual features**: loudness/music energy (`librosa`), shot detection (`PySceneDetect`) video se
3. **Emotion model**: Hinglish ke liye fine-tuned classifier ya LLM scoring
4. **FFmpeg export**: `edit_decision_list.csv` se auto rough-cut trailer render
5. **Multi-trailer variants**: Teaser (30s) / Main (90s) / Action-cut — bas `CONFIG` badlo
6. **A/B testing**: alag weights se multiple storyboards generate karke compare